# Pipeline v2: Corrected Baseline

This is a **new experiment**, separate from `pipeline_v1.ipynb` and the published original CPU/GPU runs. Set the participant ID in the next cell, then open it from the repository directory with the local data in `way_eeg_clean/`; select the WSL **Python (EEG TensorFlow GPU)** kernel or a compatible CPU kernel, then run cells in order. This notebook does not download data or install packages. It keeps the original CNN and CNN–BiLSTM architectures and epoch counts, while fixing the first evaluation problems:

- Split whole recording runs before any scaling or windowing: train 1–6, validation 7, test 8–9.
- Fit separate EEG-channel and kinematic-target scalers using **training runs only**.
- Make 128-sample EEG windows with 64-sample stride within each run; predict the following sample.
- Use validation for monitoring and report test metrics only after both fixed training schedules finish.

The selected participant’s session-start indices are one-based; this notebook converts them to Python indices. The corresponding AllLifts file confirms the run labels. The held-out runs may have a narrower set of friction conditions than the full recording, so inspect the split summary before interpreting scores. The three `kin` columns are wrist X, Y and Z, as established in [the provenance check](documentation/KINEMATICS_PROVENANCE.md). The exact before/after protocol is recorded in [the change log](documentation/PIPELINE_V2_CORRECTED_BASELINE.md). Scores from this split cannot be compared directly with the original random-window test scores.

In [ ]:
PARTICIPANT_ID = 1  # Change this to 2, 3, ..., 12 for another aligned participant.

In [ ]:
from pathlib import Path
import json, random, sys, time
from datetime import datetime, timezone
import numpy as np
import scipy
from scipy.io import loadmat
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
import matplotlib.pyplot as plt

assert isinstance(PARTICIPANT_ID, int) and PARTICIPANT_ID > 0, "PARTICIPANT_ID must be a positive integer."
PARTICIPANT = f"P{PARTICIPANT_ID}"
AXES = ("Wrist X", "Wrist Y", "Wrist Z")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
DEVICE = "gpu" if tf.config.list_physical_devices("GPU") else "cpu"
PROJECT = Path.cwd()
if not (PROJECT / "way_eeg_clean").is_dir():
    PROJECT = PROJECT / "hand_motion_eeg"
assert (PROJECT / "way_eeg_clean").is_dir(), "Open this notebook in the repository directory."
DATA = PROJECT / "way_eeg_clean"
print("TensorFlow", tf.__version__, "device", DEVICE, tf.config.list_physical_devices())

## 1. Load the selected participant’s aligned arrays and run boundaries

The cleaned EEG file is channels × samples, so transpose it to samples × channels. Kinematics already uses samples × targets. The boundary file contains the first sample of each of nine runs; MATLAB indexing starts at 1.

In [ ]:
eeg = np.asarray(loadmat(DATA / f"eeg_{PARTICIPANT}.mat", variable_names=["eeg"])["eeg"].T,
                 dtype=np.float32, order="C")
kin = np.asarray(loadmat(DATA / f"kin_{PARTICIPANT}.mat", variable_names=["kin"])["kin"],
                 dtype=np.float32, order="C")
run_starts = np.asarray(loadmat(DATA / f"session_start_{PARTICIPANT}.mat")["session_start"]).ravel().astype(np.int64) - 1
assert eeg.ndim == kin.ndim == 2, "EEG and kinematics must be two-dimensional."
assert eeg.shape[0] == kin.shape[0], f"{PARTICIPANT}: EEG has {eeg.shape[0]} samples, kinematics has {kin.shape[0]}; arrays are not aligned."
assert eeg.shape[1] == 32 and kin.shape[1] == 3, f"{PARTICIPANT}: expected 32 EEG channels and three wrist targets; got {eeg.shape[1]} and {kin.shape[1]}."
N_CHANNELS = eeg.shape[1]
assert len(run_starts) >= 4 and run_starts[0] == 0, f"{PARTICIPANT}: expected at least four runs starting at sample 1."
assert np.all(np.diff(run_starts) > 128) and run_starts[-1] < len(eeg)
assert np.isfinite(eeg).all() and np.isfinite(kin).all()
run_ends = np.r_[run_starts[1:], len(eeg)]
BOUNDS = {run: (int(start), int(end)) for run, (start, end) in enumerate(zip(run_starts, run_ends), start=1)}
lifts = loadmat(DATA / f"{PARTICIPANT}_AllLifts.mat", squeeze_me=True, struct_as_record=False)["P"]
assert set(lifts.AllLifts[:, 1].astype(int)) == set(BOUNDS)
print("EEG", eeg.shape, "kin", kin.shape)
for run, (start, end) in BOUNDS.items():
    print(f"Run {run}: samples [{start}, {end}), n={end-start:,}")

## 2. Hold out entire runs

For the supplied nine-run participants, runs 1–6 train the models, run 7 monitors validation loss, and runs 8–9 are untouched until final evaluation. The split is derived from the selected participant’s run boundaries. This split is chronological at the recording-run level and avoids overlap between train, validation and test windows.

In [ ]:
RUN_IDS = tuple(BOUNDS)
TRAIN_RUNS = RUN_IDS[:-3]
VAL_RUNS = RUN_IDS[-3:-2]
TEST_RUNS = RUN_IDS[-2:]
assert TRAIN_RUNS and VAL_RUNS and TEST_RUNS
assert not (set(TRAIN_RUNS) & set(VAL_RUNS) or set(TRAIN_RUNS) & set(TEST_RUNS) or set(VAL_RUNS) & set(TEST_RUNS))
assert set(TRAIN_RUNS + VAL_RUNS + TEST_RUNS) == set(BOUNDS)
FRICTION_BY_SPLIT = {}
LIFTS_BY_SPLIT = {}
for label, runs in [("train", TRAIN_RUNS), ("validation", VAL_RUNS), ("test", TEST_RUNS)]:
    rows = lifts.AllLifts[np.isin(lifts.AllLifts[:, 1].astype(int), runs)]
    LIFTS_BY_SPLIT[label] = int(len(rows))
    FRICTION_BY_SPLIT[label] = np.unique(rows[:, 4]).astype(int).tolist()
    print(label, "runs", runs, "samples", sum(BOUNDS[r][1] - BOUNDS[r][0] for r in runs),
          "lifts", LIFTS_BY_SPLIT[label], "friction levels", FRICTION_BY_SPLIT[label])

## 3. Fit scalers on training samples only

`StandardScaler` treats each EEG electrode and each of the three target columns separately. It is fitted incrementally on the training runs. Validation and test samples only receive the transform; they never contribute to its mean or variance. Metrics below use standardized target units, except R², which is invariant to this affine rescaling.

In [ ]:
eeg_scaler = StandardScaler()
kin_scaler = StandardScaler()
for run in TRAIN_RUNS:
    start, end = BOUNDS[run]
    eeg_scaler.partial_fit(eeg[start:end])
    kin_scaler.partial_fit(kin[start:end])
assert eeg_scaler.n_samples_seen_ == kin_scaler.n_samples_seen_ == sum(BOUNDS[r][1]-BOUNDS[r][0] for r in TRAIN_RUNS)
eeg = eeg_scaler.transform(eeg, copy=False).astype(np.float32, copy=False)
kin = kin_scaler.transform(kin, copy=False).astype(np.float32, copy=False)
assert np.isfinite(eeg).all() and np.isfinite(kin).all()
print("Scaler training samples:", eeg_scaler.n_samples_seen_)

## 4. Build windows separately within each run

For a window starting at sample `i`, the EEG input is `i:i+128`; the target is `kin[i+128]`. No window or target crosses a run boundary. Windows may overlap **within** a split, which is expected; no session appears in more than one split.

In [ ]:
WINDOW = 128
STRIDE = 64

def make_windows(run_ids):
    positions = [(run, np.arange(BOUNDS[run][0], BOUNDS[run][1] - WINDOW, STRIDE, dtype=np.int64))
                 for run in run_ids]
    total = sum(len(pos) for _, pos in positions)
    X = np.empty((total, WINDOW, eeg.shape[1]), dtype=np.float32)
    y = np.empty((total, kin.shape[1]), dtype=np.float32)
    run_id = np.empty(total, dtype=np.int8)
    target_index = np.empty(total, dtype=np.int64)
    j = 0
    for run, starts in positions:
        for start in starts:
            X[j] = eeg[start:start + WINDOW]
            y[j] = kin[start + WINDOW]
            run_id[j] = run
            target_index[j] = start + WINDOW
            j += 1
        lo, hi = BOUNDS[run]
        assert np.all((starts >= lo) & (starts + WINDOW < hi))
    return X, y, run_id, target_index

X_train, y_train, train_ids, train_targets = make_windows(TRAIN_RUNS)
X_val, y_val, val_ids, val_targets = make_windows(VAL_RUNS)
X_test, y_test, test_ids, test_targets = make_windows(TEST_RUNS)
assert len(set(train_ids) & set(val_ids)) == len(set(train_ids) & set(test_ids)) == len(set(val_ids) & set(test_ids)) == 0
assert np.isfinite(X_train).all() and np.isfinite(y_train).all()
for name, X, y, ids in [("train", X_train, y_train, train_ids), ("val", X_val, y_val, val_ids), ("test", X_test, y_test, test_ids)]:
    print(name, X.shape, y.shape, "runs", sorted(set(ids.tolist())))

## 5. Train the original architectures

The two network definitions, Adam/MSE/MAE setup, batch size 64, and 15/20 epoch counts match the original baseline. The CNN still has no nonlinear activation. No early stopping or model selection is applied, so the final epoch of each fixed schedule is evaluated.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, DepthwiseConv2D,
    SeparableConv2D, AveragePooling2D, Dropout, Flatten, Dense, Conv1D, Bidirectional, LSTM)

def make_cnn():
    model = Sequential([
        Input(shape=(WINDOW, N_CHANNELS, 1)),
        Conv2D(8, kernel_size=(64, 1), padding="same", use_bias=False),
        BatchNormalization(),
        DepthwiseConv2D((1, N_CHANNELS), use_bias=False, depth_multiplier=2, padding="valid"),
        BatchNormalization(), Dropout(0.25),
        SeparableConv2D(16, (16, 1), use_bias=False, padding="same"),
        BatchNormalization(), AveragePooling2D((4, 1)), Dropout(0.25),
        Flatten(), Dense(3),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def make_cnn_bilstm():
    model = Sequential([
        Input(shape=(WINDOW, N_CHANNELS)),
        Conv1D(filters=32, kernel_size=3, activation="relu", padding="same"),
        BatchNormalization(), Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False)), Dropout(0.3),
        Dense(64, activation="relu"), Dropout(0.2), Dense(3),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

cnn = make_cnn()
cnn_bilstm = make_cnn_bilstm()
assert cnn.count_params() == 3235 and cnn_bilstm.count_params() == 61347
print("Parameters: CNN", cnn.count_params(), "CNN–BiLSTM", cnn_bilstm.count_params())

In [ ]:
cnn_started = time.perf_counter()
cnn_history = cnn.fit(X_train[..., None], y_train, epochs=15, batch_size=64,
                      validation_data=(X_val[..., None], y_val), verbose=2)
cnn_seconds = time.perf_counter() - cnn_started
print(f"CNN training: {cnn_seconds:.2f} s")

In [ ]:
bilstm_started = time.perf_counter()
bilstm_history = cnn_bilstm.fit(X_train, y_train, epochs=20, batch_size=64,
                                validation_data=(X_val, y_val), verbose=2)
bilstm_seconds = time.perf_counter() - bilstm_started
print(f"CNN–BiLSTM training: {bilstm_seconds:.2f} s")

## 6. Evaluate the held-out test runs once

The test data have not been passed to either `fit` call. These metrics assess new runs from **the same participant**, with the friction levels shown in the split summary. They do not assess transfer to another participant or laboratory.

In [ ]:
predictions = {
    "cnn": cnn.predict(X_test[..., None], batch_size=64, verbose=0),
    "cnn_bilstm": cnn_bilstm.predict(X_test, batch_size=64, verbose=0),
}
models = {"cnn": cnn, "cnn_bilstm": cnn_bilstm}
histories = {"cnn": cnn_history, "cnn_bilstm": bilstm_history}
seconds = {"cnn": cnn_seconds, "cnn_bilstm": bilstm_seconds}
metrics = {}
for name, pred in predictions.items():
    assert pred.shape == y_test.shape and np.isfinite(pred).all()
    metrics[name] = {
        "parameters": models[name].count_params(),
        "training_seconds": float(seconds[name]),
        "mse": mean_squared_error(y_test, pred, multioutput="raw_values").tolist(),
        "mae": mean_absolute_error(y_test, pred, multioutput="raw_values").tolist(),
        "r2": r2_score(y_test, pred, multioutput="raw_values").tolist(),
        "history": {key: [float(v) for v in values] for key, values in histories[name].history.items()},
    }
    metrics[name]["by_test_run"] = {}
    for run in TEST_RUNS:
        mask = test_ids == run
        run_true, run_pred = y_test[mask], pred[mask]
        metrics[name]["by_test_run"][str(run)] = {
            "n_windows": int(mask.sum()),
            "mse": mean_squared_error(run_true, run_pred, multioutput="raw_values").tolist(),
            "mae": mean_absolute_error(run_true, run_pred, multioutput="raw_values").tolist(),
            "r2": r2_score(run_true, run_pred, multioutput="raw_values").tolist(),
        }
    print(name, "R²", np.round(metrics[name]["r2"], 4), "MAE", np.round(metrics[name]["mae"], 4))

## 7. Save this experiment separately

The output folder name records the subject, split type and device. `results.json` and the plots are suitable for review. Model weights and prediction arrays remain local under the repository's ignore rules. The original notebook and its outputs are not modified.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = PROJECT / "outputs" / f"pipeline-v2-corrected-baseline-{PARTICIPANT.lower()}-{DEVICE}-{stamp}"
run_dir.mkdir(parents=True, exist_ok=False)
results = {
    "status": "complete", "protocol": "Pipeline v2: Corrected Baseline", "subject": PARTICIPANT,
    "device": DEVICE, "seed": SEED,
    "runs": {"train": list(TRAIN_RUNS), "validation": list(VAL_RUNS), "test": list(TEST_RUNS)},
    "window_samples": WINDOW, "stride_samples": STRIDE, "target_offset_samples": WINDOW,
    "split_shapes": {"train": list(X_train.shape), "validation": list(X_val.shape), "test": list(X_test.shape)},
    "scaling": {"method": "StandardScaler", "fit_runs": list(TRAIN_RUNS),
                "eeg_mean": eeg_scaler.mean_.tolist(), "eeg_scale": eeg_scaler.scale_.tolist(),
                "target_mean": kin_scaler.mean_.tolist(), "target_scale": kin_scaler.scale_.tolist()},
    "evaluation_scope": f"{PARTICIPANT} held-out runs {TEST_RUNS}; wrist X/Y/Z targets",
    "friction_levels_by_split": FRICTION_BY_SPLIT,
    "lift_trials_by_split": LIFTS_BY_SPLIT,
    "versions": {"python": sys.version.split()[0], "tensorflow": tf.__version__,
                 "numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__},
    "devices": [str(d) for d in tf.config.list_physical_devices()],
    "models": metrics,
}
(run_dir / "results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
for name, model in models.items():
    model.save(run_dir / f"{name}.keras")
    np.savez_compressed(run_dir / f"{name}_predictions.npz", y_true=y_test, y_pred=predictions[name],
                        run_id=test_ids, target_sample_index=test_targets)
print("Saved", run_dir.relative_to(PROJECT))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
for row, name in enumerate(("cnn", "cnn_bilstm")):
    history = metrics[name]["history"]
    epochs = np.arange(1, len(history["loss"]) + 1)
    for col, (train_key, val_key, label) in enumerate((("loss", "val_loss", "MSE"), ("mae", "val_mae", "MAE"))):
        ax = axes[row, col]
        ax.plot(epochs, history[train_key], label="Train")
        ax.plot(epochs, history[val_key], label=f"Validation run(s) {VAL_RUNS}")
        ax.set(title=f"{name} · {label}", xlabel="Epoch", ylabel=f"{label} (standardized targets)")
        ax.grid(alpha=0.2); ax.legend(frameon=False)
fig.suptitle(f"Pipeline v2: Corrected Baseline · {PARTICIPANT} learning curves")
fig.savefig(run_dir / "learning_curves.png", dpi=160)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), layout="constrained")
for target, ax in enumerate(axes):
    obs = y_test[:, target]
    pred = predictions["cnn_bilstm"][:, target]
    low, high = min(obs.min(), pred.min()), max(obs.max(), pred.max())
    ax.scatter(obs, pred, s=4, alpha=0.12, rasterized=True)
    ax.plot([low, high], [low, high], "k--", linewidth=1)
    ax.set(title=f"{AXES[target]} · R² {metrics['cnn_bilstm']['r2'][target]:.3f}",
           xlabel="Observed (standardized)", ylabel="Predicted (standardized)")
    ax.grid(alpha=0.2)
fig.suptitle(f"{PARTICIPANT} test runs {TEST_RUNS} · CNN–BiLSTM predictions")
fig.savefig(run_dir / "prediction_scatter.png", dpi=160)
plt.close(fig)

lines = [f"# Pipeline v2: Corrected Baseline — {PARTICIPANT} {DEVICE.upper()}", "",
         f"Train runs {TRAIN_RUNS}; validation runs {VAL_RUNS}; held-out test runs {TEST_RUNS}. Test friction levels: {FRICTION_BY_SPLIT['test']}.",
         "Models and epoch counts match the original notebook. EEG and targets were standardized separately using training runs only.",
         "", "| Model | Training seconds | Wrist axis | MSE | MAE | R² |", "|---|---:|---:|---:|---:|---:|"]
for name in ("cnn", "cnn_bilstm"):
    m = metrics[name]
    for target in range(3):
        lines.append(f"| {name} | {m['training_seconds']:.2f} | {AXES[target]} | {m['mse'][target]:.6f} | {m['mae'][target]:.6f} | {m['r2'][target]:.6f} |")
lines += ["", "### Results by held-out run", "",
          "| Model | Run | Windows | R² wrist X | R² wrist Y | R² wrist Z |",
          "|---|---:|---:|---:|---:|---:|"]
for name in ("cnn", "cnn_bilstm"):
    for run in TEST_RUNS:
        m = metrics[name]["by_test_run"][str(run)]
        lines.append(f"| {name} | {run} | {m['n_windows']} | " + " | ".join(f"{v:.4f}" for v in m["r2"]) + " |")
lines += ["", "![Learning curves](learning_curves.png)", "", "![Prediction scatter](prediction_scatter.png)",
          "", "MSE and MAE use standardized target units. The three target columns are wrist X/Y/Z; see [kinematics provenance](../../documentation/KINEMATICS_PROVENANCE.md).",
          "Random windows from the original baseline and these held-out runs measure different tasks; compare their protocols before comparing scores.",
          "Model weights and raw predictions are local artifacts excluded from Git.", ""]
(run_dir / "RESULTS.md").write_text("\n".join(lines), encoding="utf-8")
print("Run report:", (run_dir / "RESULTS.md").relative_to(PROJECT))

## Next experiments

1. Repeat this run-level evaluation across other held-out runs for each participant, especially the mixed-friction runs, with a predeclared selection protocol.
2. After the split is stable, evaluate architecture changes such as adding nonlinear activations to the CNN.
3. For Phase 2, define population training and held-out-subject fine-tuning without allowing windows from the target subject into the population pretraining test partition.

Avoid using the test scores in this notebook to iteratively tune hyperparameters; use validation or a separate run-level cross-validation design.